In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import sys
import os

try:
    current_file_path = os.path.abspath(__file__)
    current_script_dir = os.path.dirname(current_file_path)
except NameError:
    current_script_dir = os.getcwd()

# Walk up to GlobalLocal/
project_root = os.path.abspath(os.path.join(current_script_dir, '..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import pandas as pd
import mne
import matplotlib.pyplot as plt
from types import SimpleNamespace
from functools import partial
from scipy.stats import ttest_ind, ttest_rel

from ieeg.io import get_data
from ieeg.calc.fast import mean_diff
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm
from joblib import Parallel, delayed

from src.analysis.config import experiment_conditions
from src.analysis.config.plotting_parameters import plotting_parameters as plot_params
from src.analysis.config.condition_registry import (
    get_comparisons, get_conditions_obj, get_subtraction_pairs,
    get_anova_factors, get_anova_interactions,
)
import src.analysis.utils.general_utils as utils
from src.analysis.power.power_traces import (
    make_multi_channel_evokeds_for_all_conditions_and_rois,
    process_windowed_data_for_anova,
    create_windowed_anova_dataframe,
    _fit_anova_one_window,
    _get_factor_levels,
    compute_subcell_evoked_data,
    plot_2way_interaction_for_roi,
    plot_16_conditions_with_interaction_clusters_for_roi,
)
from src.analysis.decoding.decoding import (
    find_significant_clusters_of_series_vs_distribution_based_on_percentile,
)

print("project_root:", project_root)

project_root: /hpc/group/coganlab/jz421/GlobalLocal


In [3]:
LAB_root = None  # auto-detect below
TASK = 'GlobalLocal'

SUBJECTS = ['D0103']  # start with one subject for speed; expand later
SUBJECTS = ['D0057', 'D0059', 'D0063', 'D0065', 'D0069', 'D0071', 'D0077', 'D0090', 'D0094', 'D0100', 'D0102', 'D0103', 'D0107A', 'D0110', 'D0116', 'D0117', 'D0121', 'D0133', 'D0134']

ACC_TRIALS_ONLY = False
N_JOBS = 4

CONDITION_LABEL = 'stimulus_experiment_conditions'

EPOCHS_ROOT_FILE = "Stimulus_-1.0to1.5sec_0.5sec_within-1.0-0.0sec_base_decFactor_8_outliers_10_drop_thresh_perc_5.0_70.0-150.0_Hz_padLength_0.5s_stat_func_ttest_ind_equal_var_False_nan_policy_omit"

ROIS_DICT = {
    'lpfc': ["G_front_inf-Opercular", "G_front_inf-Orbital", "G_front_inf-Triangul",
             "G_front_middle", "G_front_sup", "Lat_Fis-ant-Horizont",
             "Lat_Fis-ant-Vertical", "S_circular_insula_ant", "S_circular_insula_sup",
             "S_front_inf", "S_front_middle", "S_front_sup"],
}

ELECTRODES = 'sig'
SAMPLING_RATE = 256
WINDOW_SIZE = 64
STEP_SIZE = 16
N_PERM = 100  # small for sandbox testing; bump to 1000 for the real run

In [4]:
if LAB_root is None:
    HOME = os.path.expanduser("~")
    USER = os.path.basename(HOME)
    if os.name == 'nt':
        LAB_root = os.path.join(HOME, "Box", "CoganLab")
    elif sys.platform == 'darwin':
        LAB_root = os.path.join(HOME, "Library", "CloudStorage", "Box-Box", "CoganLab")
    else:
        LAB_root = f"/cwork/{USER}" if os.path.exists(f"/cwork/{USER}") else os.path.join(HOME, "CoganLab")
print("LAB_root:", LAB_root)

LAB_root: /cwork/jz421


In [5]:
conditions = get_conditions_obj(CONDITION_LABEL)
condition_names = list(conditions.keys())
condition_comparisons = get_comparisons(CONDITION_LABEL)
subtraction_pairs = get_subtraction_pairs(CONDITION_LABEL)
anova_factors = get_anova_factors(CONDITION_LABEL)
anova_interactions = get_anova_interactions(CONDITION_LABEL)

print("Number of conditions:", len(conditions))
print("Condition names (first 4):", condition_names[:4])
print("ANOVA factors:", anova_factors)
print("Comparisons:", condition_comparisons)
print("Subtraction pairs:", subtraction_pairs)
print("Interactions:")
for i in anova_interactions:
    print("  ", i['name'], "->", i['factors'])

Number of conditions: 16
Condition names (first 4): ['Stimulus_i25s25', 'Stimulus_i25s75', 'Stimulus_i75s25', 'Stimulus_i75s75']
ANOVA factors: ['congruency', 'congruencyProportion', 'switchType', 'switchProportion']
Comparisons: {}
Subtraction pairs: []
Interactions:
   congruency_x_congruencyProportion -> ['congruency', 'congruencyProportion']
   switchType_x_switchProportion -> ['switchType', 'switchProportion']
   congruency_x_switchProportion -> ['congruency', 'switchProportion']
   switchType_x_congruencyProportion -> ['switchType', 'congruencyProportion']


In [6]:
config_dir = os.path.join(project_root, 'src', 'analysis', 'config')
subjects_electrodestoROIs_dict = utils.make_or_load_subjects_electrodes_to_ROIs_dict(
    subjects=SUBJECTS, save_dir=config_dir,
    filename='subjects_electrodestoROIs_dict.json',
)

sig_chans_per_subject = utils.get_sig_chans_per_subject(
    SUBJECTS, EPOCHS_ROOT_FILE, task=TASK, LAB_root=LAB_root,
)

rois = list(ROIS_DICT.keys())
all_electrodes_per_subject_roi, sig_electrodes_per_subject_roi = \
    utils.make_sig_electrodes_per_subject_and_roi_dict(
        ROIS_DICT, subjects_electrodestoROIs_dict, sig_chans_per_subject,
    )

print("rois:", rois)
print("Per-ROI electrode counts (sig):")
for r in rois:
    for s in SUBJECTS:
        print(f"  {r} / {s}: {len(sig_electrodes_per_subject_roi[r].get(s, []))} sig elecs")

Attempting to load the subjects' electrodes-to-ROIs dictionary...
Loaded data from /hpc/group/coganlab/jz421/GlobalLocal/src/analysis/config/subjects_electrodestoROIs_dict.json
Dictionary loaded successfully. Ready to proceed!
Loaded significant channels for subject D0057
Loaded significant channels for subject D0059
Loaded significant channels for subject D0063
Loaded significant channels for subject D0065
Loaded significant channels for subject D0069
Loaded significant channels for subject D0071
Loaded significant channels for subject D0077
Loaded significant channels for subject D0090
Loaded significant channels for subject D0094
Loaded significant channels for subject D0100
Loaded significant channels for subject D0102
Loaded significant channels for subject D0103
Loaded significant channels for subject D0107A
Loaded significant channels for subject D0110
Loaded significant channels for subject D0116
Loaded significant channels for subject D0117
Loaded significant channels for subj

In [7]:
for roi in rois:
    print(f'total sig elecs in {roi}:', sum([len(sig_electrodes_per_subject_roi[roi][sub]) for sub in sig_electrodes_per_subject_roi[roi]]))

total sig elecs in lpfc: 139


In [8]:
subjects_mne_objects = utils.create_subjects_mne_objects_dict(
    subjects=SUBJECTS, epochs_root_file=EPOCHS_ROOT_FILE,
    conditions=conditions, task=TASK,
    just_HG_ev1_rescaled=True, acc_trials_only=ACC_TRIALS_ONLY,
)

# Inspect: what's in there?
sub0 = SUBJECTS[0]
cond0 = condition_names[0]
print("Top-level keys per subject:", list(subjects_mne_objects[sub0].keys())[:5], "...")
print(f"Per-condition keys ({sub0}/{cond0}):",
      list(subjects_mne_objects[sub0][cond0].keys()))
epochs0 = subjects_mne_objects[sub0][cond0]['HG_ev1_power_rescaled']
print("Epochs shape:", epochs0.get_data().shape, "(n_trials, n_channels, n_times)")
print("Times:", epochs0.times[0], "→", epochs0.times[-1], f"({len(epochs0.times)} samples)")

Loading data for subject: D0057
Reading /cwork/jz421/BIDS-1.1_GlobalLocal/BIDS/derivatives/freqFilt/figs/D0057/D0057_Stimulus_-1.0to1.5sec_0.5sec_within-1.0-0.0sec_base_decFactor_8_outliers_10_drop_thresh_perc_5.0_70.0-150.0_Hz_padLength_0.5s_stat_func_ttest_ind_equal_var_False_nan_policy_omit_HG_ev1_rescaled-epo.fif ...
    Found the data of interest:
        t =   -1000.00 ...    1500.00 ms
        0 CTF compensation matrices available
Adding metadata with 29 columns
449 matching events found
No baseline correction applied
0 projection items activated
Replacing existing metadata with 29 columns
Reading /cwork/jz421/BIDS-1.1_GlobalLocal/BIDS/derivatives/freqFilt/figs/D0057/D0057_Stimulus_-1.0to1.5sec_0.5sec_within-1.0-0.0sec_base_decFactor_8_outliers_10_drop_thresh_perc_5.0_70.0-150.0_Hz_padLength_0.5s_stat_func_ttest_ind_equal_var_False_nan_policy_omit_HG_ev1_power_rescaled-epo.fif ...
    Found the data of interest:
        t =   -1000.00 ...    1500.00 ms
        0 CTF compensation

/hpc/group/coganlab/jz421/GlobalLocal/src/analysis/utils/general_utils.py:511: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  event_epochs = mne.concatenate_epochs(combined_epochs_list)


  Loading condition: Stimulus_i25s75
Adding metadata with 30 columns
22 matching events found
No baseline correction applied
    Stimulus_i25s75: 22 valid trials out of 22


/hpc/group/coganlab/jz421/GlobalLocal/src/analysis/utils/general_utils.py:511: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  event_epochs = mne.concatenate_epochs(combined_epochs_list)


  Loading condition: Stimulus_i75s25
Adding metadata with 30 columns
23 matching events found
No baseline correction applied
    Stimulus_i75s25: 23 valid trials out of 23


...

/hpc/group/coganlab/jz421/GlobalLocal/src/analysis/utils/general_utils.py:511: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  event_epochs = mne.concatenate_epochs(combined_epochs_list)


    Stimulus_c25r75: 22 valid trials out of 22
  Loading condition: Stimulus_c75r25
Adding metadata with 30 columns
21 matching events found
No baseline correction applied


/hpc/group/coganlab/jz421/GlobalLocal/src/analysis/utils/general_utils.py:511: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  event_epochs = mne.concatenate_epochs(combined_epochs_list)


    Stimulus_c75r25: 21 valid trials out of 21
  Loading condition: Stimulus_c75r75
Adding metadata with 30 columns
4 matching events found
No baseline correction applied
    Stimulus_c75r75: 4 valid trials out of 4
Top-level keys per subject: ['Stimulus_i25s25', 'Stimulus_i25s75', 'Stimulus_i75s25', 'Stimulus_i75s75', 'Stimulus_i25r25'] ...
Per-condition keys (D0057/Stimulus_i25s25): ['HG_ev1_rescaled', 'HG_ev1_rescaled_avg', 'HG_ev1_rescaled_std_err', 'HG_ev1_power_rescaled', 'HG_ev1_power_rescaled_avg', 'HG_ev1_power_rescaled_std_err']
Epochs shape: (7, 175, 641) (n_trials, n_channels, n_times)
Times: -1.0 → 1.5 (641 samples)


/hpc/group/coganlab/jz421/GlobalLocal/src/analysis/utils/general_utils.py:511: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  event_epochs = mne.concatenate_epochs(combined_epochs_list)


In [9]:
if ELECTRODES == 'all':
    raw_electrodes = all_electrodes_per_subject_roi
    elec_string_to_add_to_filename = 'all_elecs'
else:
    raw_electrodes = sig_electrodes_per_subject_roi
    elec_string_to_add_to_filename = 'sig_elecs'

electrodes = utils.filter_electrode_lists_against_subjects_mne_objects(
    rois, raw_electrodes, subjects_mne_objects,
)

dropped, _ = utils.find_difference_between_two_electrode_lists(raw_electrodes, electrodes)
total_dropped = sum(len(v) for d in dropped.values() for v in d.values())
print(f"Total dropped electrodes: {total_dropped}")
print("Final electrodes per ROI per subject:")
for r in rois:
    for s in SUBJECTS:
        print(f"  {r} / {s}: {len(electrodes[r].get(s, []))} electrodes")

Total dropped electrodes: 0
Final electrodes per ROI per subject:
  lpfc / D0057: 5 electrodes
  lpfc / D0059: 5 electrodes
  lpfc / D0063: 8 electrodes
  lpfc / D0065: 1 electrodes
  lpfc / D0069: 0 electrodes
  lpfc / D0071: 10 electrodes
  lpfc / D0077: 0 electrodes
  lpfc / D0090: 2 electrodes
  lpfc / D0094: 10 electrodes
  lpfc / D0100: 0 electrodes
  lpfc / D0102: 3 electrodes
  lpfc / D0103: 9 electrodes
  lpfc / D0107A: 9 electrodes
  lpfc / D0110: 2 electrodes
  lpfc / D0116: 19 electrodes
  lpfc / D0117: 3 electrodes
  lpfc / D0121: 17 electrodes
  lpfc / D0133: 17 electrodes
  lpfc / D0134: 19 electrodes


In [10]:
evks_dict_elecs = make_multi_channel_evokeds_for_all_conditions_and_rois(
    subjects_mne_objects, SUBJECTS, rois, condition_names, electrodes,
)

# Inspect
roi0 = rois[0]
cond0 = condition_names[0]
evk = evks_dict_elecs[cond0][roi0]
print(f"Evoked for {cond0} / {roi0}:")
print(f"  data shape: {evk.data.shape}  (n_electrodes, n_times)")
print(f"  ch_names (first 3): {evk.ch_names[:3]}")
print(f"  times: {evk.times[0]:.3f} → {evk.times[-1]:.3f}")

NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy functi

NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy functi

/hpc/group/coganlab/jz421/GlobalLocal/src/analysis/power/power_traces.py:124: RuntimeWarning: Channel names are not unique, found duplicates for: {'LAI7', 'LFAI9', 'LFAM9', 'LFAM10', 'RFO14', 'RAI6', 'LFAM8', 'LFAI4', 'LFMM8', 'LFAI10', 'LFMM10', 'RIA11', 'LFMM9', 'LAI8', 'RIA6'}. Applying running numbers for duplicates.
  info = mne.create_info(
/hpc/group/coganlab/jz421/GlobalLocal/src/analysis/power/power_traces.py:124: RuntimeWarning: Channel names are not unique, found duplicates for: {'LAI7', 'LFAI9', 'LFAM9', 'LFAM10', 'RFO14', 'RAI6', 'LFAM8', 'LFAI4', 'LFMM8', 'LFAI10', 'LFMM10', 'RIA11', 'LFMM9', 'LAI8', 'RIA6'}. Applying running numbers for duplicates.
  info = mne.create_info(
/hpc/group/coganlab/jz421/GlobalLocal/src/analysis/power/power_traces.py:124: RuntimeWarning: Channel names are not unique, found duplicates for: {'LAI7', 'LFAI9', 'LFAM9', 'LFAM10', 'RFO14', 'RAI6', 'LFAM8', 'LFAI4', 'LFMM8', 'LFAI10', 'LFMM10', 'RIA11', 'LFMM9', 'LAI8', 'RIA6'}. Applying running num

In [11]:
windowed_data = process_windowed_data_for_anova(
    subjects_mne_objects, condition_names, rois, SUBJECTS,
    electrodes, window_size=WINDOW_SIZE,
    step_size=STEP_SIZE, sampling_rate=SAMPLING_RATE,
)

# Inspect
arr0 = windowed_data[cond0][roi0][0]  # subject 0
print(f"windowed_data[{cond0}][{roi0}][sub0] shape:")
print(f"  {arr0.shape}  (n_trials, n_windows, n_channels)")
n_windows_expected = (len(evk.times) - WINDOW_SIZE) // STEP_SIZE + 1
print(f"  expected n_windows: {n_windows_expected}")

NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy functi

In [12]:
ref_evk = next(
    (evks_dict_elecs[c][r] for c in condition_names for r in rois
     if evks_dict_elecs[c][r] is not None and evks_dict_elecs[c][r].data.shape[0] > 0),
    None,
)

full_times = ref_evk.times

df = create_windowed_anova_dataframe(
    windowed_data, conditions, rois, electrodes_per_subject_roi=electrodes, subjects=SUBJECTS,
    times=full_times, window_size=WINDOW_SIZE,
    step_size=STEP_SIZE, sampling_rate=SAMPLING_RATE,
)

print("DataFrame shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nHead:")
df.head()

DataFrame shape: (2287044, 12)
Columns: ['SubjectID', 'Electrode', 'ROI', 'WindowCenter', 'WindowIndex', 'Trial', 'Activity', 'BIDS_events', 'congruency', 'congruencyProportion', 'switchType', 'switchProportion']

Head:


,SubjectID,Electrode,ROI,WindowCenter,WindowIndex,Trial,Activity,BIDS_events,congruency,congruencyProportion,switchType,switchProportion
0,D0057,RAI6,lpfc,-0.875,0,1,0.081651,[Stimulus/i25.0/s25.0],i,75%,s,25%
1,D0057,RPI15,lpfc,-0.875,0,1,-0.530250,[Stimulus/i25.0/s25.0],i,75%,s,25%
2,D0057,RPI14,lpfc,-0.875,0,1,1.090810,[Stimulus/i25.0/s25.0],i,75%,s,25%
3,D0057,RAMF13,lpfc,-0.875,0,1,-0.819367,[Stimulus/i25.0/s25.0],i,75%,s,25%
4,D0057,RAMF14,lpfc,-0.875,0,1,-0.156300,[Stimulus/i25.0/s25.0],i,75%,s,25%


In [53]:
data_for_anova = []

# Window centers
if WINDOW_SIZE is not None:
    n_windows = (len(full_times) - WINDOW_SIZE) // STEP_SIZE + 1
    window_centers = []
    for i in range(n_windows):
        start_idx = i * STEP_SIZE
        center_idx = start_idx + WINDOW_SIZE // 2
        if center_idx < len(full_times):
            window_centers.append(full_times[center_idx])
        else:
            window_centers.append(full_times[-1])
else:
    window_centers = [np.mean(full_times)]
    
window_centers

[np.float64(-0.875),
 np.float64(-0.8125),
 np.float64(-0.75),
 np.float64(-0.6875),
 np.float64(-0.625),
 np.float64(-0.5625),
 np.float64(-0.5),
 np.float64(-0.4375),
 np.float64(-0.375),
 np.float64(-0.3125),
 np.float64(-0.25),
 np.float64(-0.1875),
 np.float64(-0.125),
 np.float64(-0.0625),
 np.float64(0.0),
 np.float64(0.0625),
 np.float64(0.125),
 np.float64(0.1875),
 np.float64(0.25),
 np.float64(0.3125),
 np.float64(0.375),
 np.float64(0.4375),
 np.float64(0.5),
 np.float64(0.5625),
 np.float64(0.625),
 np.float64(0.6875),
 np.float64(0.75),
 np.float64(0.8125),
 np.float64(0.875),
 np.float64(0.9375),
 np.float64(1.0),
 np.float64(1.0625),
 np.float64(1.125),
 np.float64(1.1875),
 np.float64(1.25),
 np.float64(1.3125),
 np.float64(1.375)]

In [61]:
test1 = {'col1': {'nested_1': 1, 'nested_2': 2}, 'col2': 2}
test2 = {'col1': 3}

test3 = [test1, test2]
x = pd.DataFrame.from_dict(test1)
x

,col1,col2
nested_1,1,2
nested_2,2,2


In [54]:
for condition_name, condition_parameters in conditions.items():
    for roi in rois:
        roi_list = windowed_data.get(condition_name, {}).get(roi, [])
        sub_idx = 0
        for sub in SUBJECTS:
            electrodes_test = electrodes[roi].get(sub, [])
            if not electrodes_test:
                continue  # process_windowed_data_for_anova also skipped
            if sub_idx >= len(roi_list):
                break
            subject_data = roi_list[sub_idx]
            sub_idx += 1

            # Defensive: clamp electrode count to actual data shape.
            n_chans_data = subject_data.shape[2]
            n_chans = min(len(electrodes_test), n_chans_data)
            if n_chans_data != len(electrodes_test):
                print(f"[create_windowed_anova_dataframe] WARNING: "
                        f"{sub}/{roi}: electrode list has {len(electrodes_test)} but "
                        f"data has {n_chans_data} channels; using first {n_chans}.")

            for trial_idx in range(subject_data.shape[0]):
                for window_idx in range(subject_data.shape[1]):
                    for electrode_idx in range(n_chans):
                        electrode_name = electrodes_test[electrode_idx]
                        activity = subject_data[trial_idx, window_idx, electrode_idx]
                        data_dict = {
                            'SubjectID': sub,
                            'Electrode': electrode_name,
                            'ROI': roi,
                            'WindowCenter': window_centers[window_idx],
                            'WindowIndex': window_idx,
                            'Trial': trial_idx + 1,
                            'Activity': activity,
                        }
                        data_dict.update(condition_parameters)
                        data_for_anova.append(data_dict)

x = pd.DataFrame(data_for_anova)

In [ ]:
def process_windowed_data_for_anova(subjects_mne_objects, condition_names, rois, subjects, 
                                    electrodes_per_subject_roi, window_size=64, 
                                    step_size=16, sampling_rate=256):
    """
    Process data with sliding windows for ANOVA analysis.
    
    Parameters:
    -----------
    window_size : int or None
        Size of window in samples. If None, uses full epoch.
    step_size : int
        Step size for sliding window in samples.
    Slide a window over each subject's trial-level epoch data and average within
    each window to produce per-window per-channel scalars suitable for ANOVA.

    For every (condition, ROI, subject) triple that has at least one electrode in
    the ROI, this function:
      1. Picks the subject's epochs for the given condition.
      2. Restricts channels to ``electrodes_per_subject_roi[roi][subject]``.
      3. Slides a window of ``window_size`` samples with stride ``step_size`` over
         the time axis using ``general_utils.windower``.
      4. Averages within each window (collapsing the time axis inside a window)
         to produce a (n_trials, n_windows, n_channels) array.

    Subjects with **no** electrodes for the ROI are silently skipped (the output
    list for that ROI simply omits them). The downstream
    ``create_windowed_anova_dataframe`` reproduces this same skipping logic so
    list indices line up with the right subject + electrodes — see
    ``create_windowed_anova_dataframe`` for the matching invariant.

    Parameters
    ----------
    subjects_mne_objects : dict
        Nested dict ``[subject][condition_name][mne_object_type]``; we read
        ``[sub][cond]['HG_ev1_power_rescaled']``.
    condition_names : list of str
        Condition keys (e.g. ``['Stimulus_i25s25', ...]``).
    rois : list of str
        ROI names (e.g. ``['lpfc', 'occ', ...]``).
    subjects : list of str
        Subject IDs to iterate. Must be the same ordered list passed to
        ``create_windowed_anova_dataframe`` later, since that function relies on
        the same subject-skipping order.
    electrodes_per_subject_roi : dict
        ``[roi][subject] -> list of electrode names``.
    window_size : int or None, default 64
        Size of window in samples. If None, ``windower`` falls back to a single
        full-epoch window.
    step_size : int, default 16
        Stride between successive windows in samples.
    sampling_rate : int, default 256
        Currently unused (kept for API symmetry with the dataframe builder).

    Returns
    -------
    windowed_data : dict
        ``{condition_name: {roi: list_of_arrays}}`` where each list entry is a
        (n_trials, n_windows, n_channels) ndarray, one per subject that had
        electrodes for that ROI, in the same order as ``subjects``.
    """
    windowed_data = {}
    
    for condition_name in condition_names:
        windowed_data[condition_name] = {}
        
        for roi in rois:
            roi_data_list = []
            
            for sub in subjects:
                electrodes = electrodes_per_subject_roi[roi].get(sub, [])
                if not electrodes:
                    continue
                
                # Get epochs for this condition
                epochs = subjects_mne_objects[sub][condition_name]['HG_ev1_power_rescaled'].copy()
                epochs = epochs.pick_channels(electrodes)
                
                # Get data: (n_trials, n_channels, n_times)
                data = epochs.get_data()
                
                # Apply windowing to each channel and trial
                # windower expects data with time axis last by default
                windowed_trials = []
                for trial in data:
                    trial_windowed = windower(trial, window_size=window_size, 
                                             axis=-1, step_size=step_size, insert_at=0)
                    # Shape: (n_windows, n_channels, window_size)
                    windowed_trials.append(trial_windowed)
                
                windowed_trials = np.array(windowed_trials)
                # Shape: (n_trials, n_windows, n_channels, window_size)
                
                # Average within each window
                windowed_avg = np.mean(windowed_trials, axis=-1)
                # Shape: (n_trials, n_windows, n_channels)
                
                roi_data_list.append(windowed_avg)
            
            windowed_data[condition_name][roi] = roi_data_list
    
    return windowed_data

def create_windowed_anova_dataframe(windowed_data, conditions, rois, subjects,
                                    electrodes_per_subject_roi, times,
                                    window_size=None, step_size=1, sampling_rate=256):
    """
    Create DataFrame for windowed ANOVA analysis.
    Flatten the nested ``windowed_data`` structure into a long DataFrame suitable
    for per-window OLS / ANOVA fits.

    Each row is one observation: ``(subject, electrode, ROI, trial, window) ->
    Activity``, with the factor columns from ``conditions[cond]`` attached
    (e.g. ``congruency``, ``congruencyProportion``, ``switchType``,
    ``switchProportion``). ``BIDS_events`` is included verbatim; downstream
    formula-builders should drop it.

    Window centers are precomputed once (in seconds) and attached to every row
    of the corresponding window so the resulting DataFrame can be plotted /
    grouped by ``WindowCenter`` directly.

    Parameters
    ----------
    windowed_data : dict
        Output of :func:`process_windowed_data_for_anova`.
    conditions : dict
        Mapping ``{condition_name: condition_parameters}`` from
        ``experiment_conditions``. The ``condition_parameters`` dict is merged
        into every row so factor columns become available for OLS.
    rois : list of str
    electrodes_per_subject_roi : dict
        ``[roi][subject] -> list of electrode names``.
    times : array-like
        The full per-sample time vector of the epoch (used to compute window
        centers).
    window_size : int or None
        Window length in samples; when None, a single window centered at the
        epoch midpoint is produced.
    step_size : int, default 1
    sampling_rate : int, default 256

    Returns
    -------
    df : pandas.DataFrame
        Columns: ``SubjectID``, ``Electrode``, ``ROI``, ``WindowCenter``,
        ``WindowIndex``, ``Trial``, ``Activity``, plus all keys from
        ``condition_parameters`` (factor columns + ``BIDS_events``).

    Notes
    -----
    Cross-subject channel-name collisions: ``combine_single_channel_evokeds``
    renames duplicates with running suffixes (``LFMM9-0``, ``LFMM9-1``), but the
    ``Electrode`` column here stores the **un-suffixed** original name. Always
    group by ``(SubjectID, Electrode)`` together if you need a unique key.
    """
    data_for_anova = []

    # Window centers
    if window_size is not None:
        n_windows = (len(times) - window_size) // step_size + 1
        window_centers = []
        for i in range(n_windows):
            start_idx = i * step_size
            center_idx = start_idx + window_size // 2
            if center_idx < len(times):
                window_centers.append(times[center_idx])
            else:
                window_centers.append(times[-1])
    else:
        window_centers = [np.mean(times)]

    for condition_name, condition_parameters in conditions.items():
        for roi in rois:
            roi_list = windowed_data.get(condition_name, {}).get(roi, [])
            sub_idx = 0
            for sub in subjects:
                electrodes = electrodes_per_subject_roi[roi].get(sub, [])
                if not electrodes:
                    continue  # process_windowed_data_for_anova also skipped
                if sub_idx >= len(roi_list):
                    break
                subject_data = roi_list[sub_idx]
                sub_idx += 1

                # Defensive: clamp electrode count to actual data shape.
                n_chans_data = subject_data.shape[2]
                n_chans = min(len(electrodes), n_chans_data)
                if n_chans_data != len(electrodes):
                    print(f"[create_windowed_anova_dataframe] WARNING: "
                          f"{sub}/{roi}: electrode list has {len(electrodes)} but "
                          f"data has {n_chans_data} channels; using first {n_chans}.")

                for trial_idx in range(subject_data.shape[0]):
                    for window_idx in range(subject_data.shape[1]):
                        for electrode_idx in range(n_chans):
                            electrode_name = electrodes[electrode_idx]
                            activity = subject_data[trial_idx, window_idx, electrode_idx]
                            data_dict = {
                                'SubjectID': sub,
                                'Electrode': electrode_name,
                                'ROI': roi,
                                'WindowCenter': window_centers[window_idx],
                                'WindowIndex': window_idx,
                                'Trial': trial_idx + 1,
                                'Activity': activity,
                            }
                            data_dict.update(condition_parameters)
                            data_for_anova.append(data_dict)

    return pd.DataFrame(data_for_anova)

def perform_windowed_anova(df, conditions, rois, save_dir, save_name, 
                           anova_type='within_electrode'):
    """
    Fit a Type II OLS ANOVA at every time window and persist the results.

    The OLS formula is built dynamically from the keys of any single condition
    in ``conditions`` (excluding ``BIDS_events``), as
    ``Activity ~ C(f1) * C(f2) * ... * C(fn)`` — i.e. all factors plus all
    interactions. This implicitly assumes that every condition shares the same
    factor-key set; mixing condition sets with different keys here will produce
    misleading formulas.

    Two analysis modes:

    - ``'within_electrode'``: per (subject, electrode, ROI), fit OLS on the
      trial-level rows for that electrode in that window. Stores only the
      effects with uncorrected p < 0.05.
    - ``'across_electrode'``: per ROI, average activity within each
      (subject, electrode, factors) cell first, then fit OLS treating each
      electrode-cell as one observation. Stores the full ANOVA table per ROI
      per window.

    Parameters
    ----------
    df : pandas.DataFrame
        Output of :func:`create_windowed_anova_dataframe`.
    conditions : dict
        ``{condition_name: condition_parameters}``; only the first entry's keys
        are used to build the formula.
    rois : list of str
        Used only in the ``'across_electrode'`` branch.
    save_dir : str
        Directory to write the JSON output file.
    save_name : str
        Stem for the output filename
        (``{save_name}_windowed_anova_{anova_type}.json``).
    anova_type : {'within_electrode', 'across_electrode'}, default 'within_electrode'

    Returns
    -------
    results_by_window : dict
        - within_electrode: ``{window_center -> list of {SubjectID, Electrode,
          ROI, Effects (DataFrame of significant effects)}}``
        - across_electrode: ``{window_center -> {roi: anova_lm_table}}``

    Notes
    -----
    BUG (across_electrode branch): the inner loop overwrites
    ``results_by_window[window_center]`` for each ROI, so only the last ROI's
    table is preserved per window. To persist all ROIs, that branch should
    accumulate into ``{window_center: {roi: ...}}`` instead of reassigning.

    Output JSON serializes via ``str(result)``, which is lossy and only useful
    for human inspection — re-parsing the JSON back into DataFrames is not
    supported.
    """
    results_by_window = {}
    
    # Get unique window indices
    window_indices = df['WindowIndex'].unique()
    
    for window_idx in window_indices:
        df_window = df[df['WindowIndex'] == window_idx]
        window_center = df_window['WindowCenter'].iloc[0]
        
        if anova_type == 'within_electrode':
            # Perform within-electrode ANOVA for this window
            results = []
            
            for subject_id in df_window['SubjectID'].unique():
                for electrode in df_window['Electrode'].unique():
                    for roi in df_window['ROI'].unique():
                        df_filtered = df_window[
                            (df_window['SubjectID'] == subject_id) & 
                            (df_window['Electrode'] == electrode) & 
                            (df_window['ROI'] == roi)
                        ]
                        
                        if df_filtered.empty or len(df_filtered) < 2:
                            continue
                        
                        # Build formula
                        condition_keys = [k for k in conditions[next(iter(conditions))].keys() 
                                        if k != 'BIDS_events']
                        formula = 'Activity ~ ' + ' * '.join([f'C({k})' for k in condition_keys])
                        
                        try:
                            model = ols(formula, data=df_filtered).fit()
                            anova_results = anova_lm(model, typ=2)
                            
                            # Store significant effects
                            sig_effects = anova_results[anova_results['PR(>F)'] < 0.05]
                            if not sig_effects.empty:
                                results.append({
                                    'SubjectID': subject_id,
                                    'Electrode': electrode,
                                    'ROI': roi,
                                    'Effects': sig_effects
                                })
                        except:
                            continue
            
            results_by_window[window_center] = results
            
        elif anova_type == 'across_electrode':
            # Perform across-electrode ANOVA for this window.
            # Accumulate per-ROI results under this window so multiple ROIs
            # don't overwrite each other.
            if window_center not in results_by_window:
                results_by_window[window_center] = {}

            # Build the factor-key list once per window — exclude BIDS_events
            # because (a) it isn't an ANOVA factor and (b) its values are lists,
            # which are unhashable and would break the groupby below.
            condition_keys = [k for k in conditions[next(iter(conditions))].keys()
                              if k != 'BIDS_events']
            formula = 'Activity ~ ' + ' * '.join([f'C({k})' for k in condition_keys])

            for roi in rois:
                df_roi = df_window[df_window['ROI'] == roi]
                if df_roi.empty:
                    continue

                # Average across trials for each electrode-cell
                df_averaged = df_roi.groupby(
                    ['SubjectID', 'Electrode', 'ROI'] + condition_keys
                )['Activity'].mean().reset_index()

                model = ols(formula, data=df_averaged).fit()
                anova_results = anova_lm(model, typ=2)

                results_by_window[window_center][roi] = anova_results
    
    # Save results
    results_file = os.path.join(save_dir, f'{save_name}_windowed_anova_{anova_type}.json')
    
    # Convert results to serializable format
    serializable_results = {}
    for window, result in results_by_window.items():
        serializable_results[str(window)] = str(result)
    
    with open(results_file, 'w') as f:
        json.dump(serializable_results, f, indent=4)
    
    return results_by_window


In [ ]:
def main(args):





    # ------------------------------------------------------------------
    # 4. Build evokeds for all conditions
    # ------------------------------------------------------------------
    evks_dict_elecs = make_multi_channel_evokeds_for_all_conditions_and_rois(
        subjects_mne_objects, args.subjects, rois, condition_names, electrodes
    )

    # ------------------------------------------------------------------
    # 5. Statistical testing
    # ------------------------------------------------------------------
    significant_clusters = {}
    interaction_results = None

    if args.statistical_method == 'time_perm_cluster':
        print("\nRunning statistical tests comparing TWO conditions")
        if len(condition_names) != 2:
            raise ValueError("Time perm cluster stats requires exactly two conditions.")

        p_values_dict = {}
        condition1_name, condition2_name = condition_names

        for roi in rois:
            print(f"-- Processing ROI: {roi} --")
            try:
                evoked_cond1_this_roi = evks_dict_elecs[condition1_name][roi]
                evoked_cond2_this_roi = evks_dict_elecs[condition2_name][roi]

                mask_roi, p_values = time_perm_cluster_between_two_evokeds(
                    evoked_cond1_this_roi, evoked_cond2_this_roi,
                    p_thresh=args.p_thresh_for_time_perm_cluster_stats,
                    p_cluster=args.p_cluster, n_perm=args.n_perm,
                    tails=args.tails, axis=0, stat_func=args.stat_func,
                    ignore_adjacency=None,
                    permutation_type=args.permutation_type,
                    vectorized=True, n_jobs=args.n_jobs, seed=None, verbose=True,
                )

                significant_clusters[roi] = mask_roi
                p_values_dict[roi] = p_values
            except KeyError:
                print(f"   Skipping ROI {roi}: Missing prepared evoked data for "
                      f"one or both conditions.")

    elif args.statistical_method == 'anova':
        if not anova_interactions:
            raise ValueError(
                f"condition_label '{condition_label}' has no 'anova_interactions' "
                f"in the registry. Add them to enable ANOVA cluster correction."
            )
        print(f"\nRunning ANOVA-based 2-way interaction cluster correction for "
              f"{len(anova_interactions)} interaction(s) across {len(rois)} ROI(s)")
        interaction_results = run_anova_interaction_clusters(
            evks_dict_elecs, conditions, anova_interactions, rois,
            p_thresh=args.p_thresh_for_time_perm_cluster_stats,
            cluster_forming_p=args.p_cluster,
            n_perm=args.n_perm, tails=args.tails,
            seed=None, verbose=True,
        )

    # ------------------------------------------------------------------
    # 6. Plotting
    # ------------------------------------------------------------------
    plot_power_traces_for_all_rois(
        evks_dict_elecs, rois, condition_names, conditions_save_name, plot_params,
        args.window_size, args.sampling_rate,
        significant_clusters=significant_clusters or None,
        save_dir=save_dir,
        error_type='sem',
        plot_style=args.plot_style,
        save_name_suffix=elec_string_to_add_to_filename,
    )

    # If we ran the ANOVA cluster correction, draw the 16-condition mega-plot
    # with 4 stacked interaction-cluster bars + one 4-trace plot per interaction.
    if interaction_results is not None:
        plot_anova_interaction_results(
            evks_dict_elecs, conditions, condition_names, conditions_save_name,
            rois, anova_interactions, interaction_results,
            plot_style=args.plot_style, save_dir=save_dir,
            save_name_suffix=elec_string_to_add_to_filename, error_type='sem',
        )

    # ------------------------------------------------------------------
    # 7. Subtraction-pair plotting (driven by registry)
    # ------------------------------------------------------------------
    if subtraction_pairs:
        # All names needed must be present in evks_dict_elecs
        usable_pairs = [p for p in subtraction_pairs
                        if p[0] in evks_dict_elecs and p[1] in evks_dict_elecs]
        if usable_pairs:
            subtracted_evks_dict_elecs = create_subtracted_evokeds_dict(
                evks_dict_elecs, usable_pairs, rois
            )
            sub_pair_names = ['-'.join(pair) for pair in usable_pairs]
            sub_save_name = (f"{condition_label}_subtractions_"
                             f"{args.epochs_root_file}_"
                             f"{len(args.subjects)}_subjects")
            plot_power_traces_for_all_rois(
                subtracted_evks_dict_elecs, rois, sub_pair_names, sub_save_name,
                plot_params, save_dir=save_dir,
                window_size=args.window_size, sampling_rate=args.sampling_rate,
                error_type='sem',
                plot_style=args.plot_style,
                save_name_suffix=elec_string_to_add_to_filename,
            )
        else:
            print("[subtraction] No usable subtraction pairs (condition keys "
                  "missing from evks_dict).")

    # ------------------------------------------------------------------
    # 8. Save results
    # ------------------------------------------------------------------
    results_save_dir = os.path.join(save_dir, 'saved_results')
    os.makedirs(results_save_dir, exist_ok=True)

    for condition_name in condition_names:
        for roi in rois:
            evk = evks_dict_elecs[condition_name][roi]
            if evk is not None:
                np.savez(
                    os.path.join(
                        results_save_dir,
                        f'{conditions_save_name}_{condition_name}_{roi}_evoked.npz'
                    ),
                    data=evk.data, times=evk.times, ch_names=evk.ch_names
                )

    if significant_clusters:
        np.savez(
            os.path.join(results_save_dir,
                         f'{conditions_save_name}_significant_clusters.npz'),
            **significant_clusters
        )

    if interaction_results is not None:
        # Persist as one file per ROI / interaction (masks + p values)
        for roi, by_inter in interaction_results.items():
            for inter_name, info in by_inter.items():
                np.savez(
                    os.path.join(
                        results_save_dir,
                        f'{conditions_save_name}_{roi}_{inter_name}_interaction_cluster.npz'
                    ),
                    mask=info['mask'],
                    t_obs=info['t_obs'],
                    cluster_p_values=info['cluster_p_values'],
                )

    metadata = {
        'condition_label': condition_label,
        'condition_names': condition_names,
        'rois': rois,
        'conditions_save_name': conditions_save_name,
        'sfreq': args.sampling_rate,
        'statistical_method': args.statistical_method,
        'anova_factors': anova_factors,
        'anova_interactions': [
            {'name': i['name'], 'factors': i['factors'],
             'label': i.get('label', i['name'])}
            for i in anova_interactions
        ] if anova_interactions else [],
    }
    with open(os.path.join(results_save_dir,
                           f'{conditions_save_name}_metadata.json'), 'w') as f:
        json.dump(metadata, f, indent=2)


if __name__ == "__main__":
    if len(sys.argv) == 1:
        pass
    else:
        print("This script should be called via run_power_traces_dcc.py")
        print("Direct command-line execution is not supported with complex parameters.")
        sys.exit(1)